# spark.read — read a CSV (try it live)

Runnable companion to the note: **[DataFrameReader](https://ravi-writes.pages.dev/notes/abinitio-to-pyspark/core-operations/dataframereader)**.

`spark.read` is Ab Initio's **Input File / Input Table**. This example is self-contained but uses a **real file**: it first *writes* a small CSV into Colab's `/content`, then *reads* it back. Run the cells top to bottom.

## The Ab Initio equivalent

`spark.read` = **Input File**, `df.write` = **Output File**. We write a file (setup), then read it — two components joined by the same CSV on disk.

> **Two things to notice about files.** `/content` is Colab's scratch disk — **ephemeral**, wiped when the runtime recycles. And Spark's `write` produces a **directory of `part-*` files** (plus a `_SUCCESS` marker), not a single `customers.csv` — like an Ab Initio multifile / partitioned output.

```text
  Create Data ──(df.write)──►  Output File  /content/customers
                                    ┆  (same CSV on disk)
                                    ▼
  Input File  ──(spark.read)──►  DataFrame
```

## 0. Install PySpark

Colab doesn't ship with Spark — this one-time `pip install` sets it up in the runtime (a few seconds).

In [ ]:
!pip install -q pyspark

## 1. Imports & SparkSession

`SparkSession.builder...getOrCreate()` is the builder object — see [The Builder Object](https://ravi-writes.pages.dev/notes/python-for-spark/).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType,
)

spark = SparkSession.builder.appName("read-demo").getOrCreate()

## 2. Write a CSV into /content (setup)

Generate a few rows and write them out — this is the file we'll read back. Spark writes a **directory** of `part-*.csv` files.

In [ ]:
seed = spark.createDataFrame(
    [
        (1, "Asha", "2024-03-01", 120.0),
        (2, "Ben",  "2023-11-05",   0.0),
        (3, "Chen", "2024-01-15", 300.0),
    ],
    ["id", "name", "signup_date", "balance"],
)

seed.write.mode("overwrite").option("header", True).csv("/content/customers")

## 3. Read it back with spark.read

With no schema, every CSV column comes in as a **string**. Pin the types with an explicit schema — the Ab Initio DML parallel.

In [ ]:
# no schema -> every column is a string
raw = spark.read.option("header", True).csv("/content/customers")
raw.printSchema()

# explicit schema -> exact types (the Ab Initio DML)
schema = StructType([
    StructField("id",          IntegerType(), nullable=True),
    StructField("name",        StringType(),  nullable=True),
    StructField("signup_date", StringType(),  nullable=True),
    StructField("balance",     DoubleType(),  nullable=True),
])

df = spark.read.schema(schema).option("header", True).csv("/content/customers")
df.orderBy("id").show()   # a file read has no guaranteed row order — sort it
df.printSchema()

## Your turn

1. **Let Spark guess** — read again with `.option("inferSchema", True)` (and no `.schema(...)`), then `printSchema()`. What did it get right? (`inferSchema` is Spark's convenience; Ab Initio always needs its DML up front.)
2. **The generic form** — rewrite the typed read as `spark.read.format("csv").schema(schema).option("header", True).load("/content/customers")` and confirm it matches.
3. **What got written?** — `import os; print(os.listdir("/content/customers"))` — you'll see `part-*.csv` files and a `_SUCCESS` marker: a **directory**, not one file.

Try them yourself first, then reveal the solutions below.

In [ ]:
# 1. Let Spark infer the types
inferred = spark.read.option("header", True).option("inferSchema", True).csv("/content/customers")
inferred.printSchema()

# 2. Generic form — format + load — same result as the .csv(...) shortcut
df2 = spark.read.format("csv").schema(schema).option("header", True).load("/content/customers")
df2.orderBy("id").show()

# 3. Spark wrote a DIRECTORY of part files, not a single file
import os
print(os.listdir("/content/customers"))